# 05 — DORA Vendor Risk Propagation (PyReason)

Demonstrates PyReason adapter for DORA ICT vendor compliance:
- Boolean risk labels propagating across a vendor dependency graph
- Temporal propagation over multiple timesteps
- Uncertainty/severity kept as side-channel audit context (not PyReason payloads)

Uses the **adapter-local session + runner** path (not `Store.evaluate`). This gives
direct control over the PyReason session, rules, and timestep configuration.

**Prerequisites:** [01](01_sdk_basics.ipynb)–[02](02_rules_and_derivations.ipynb).  
**Requires:** `pyreason==3.0.0` (graceful fallback if not installed).  
**Next:** [06_problog_probabilistic.ipynb](06_problog_probabilistic.ipynb)

In [7]:
from __future__ import annotations
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from kernel.adapters.pyreason.accept import accept_pyreason_session
from kernel.adapters.pyreason.rule_ext import PyReasonRuleExt
from kernel.adapters.pyreason.runner import PyReasonRunConfig, run_pyreason
from kernel.adapters.pyreason.session import PyReasonSession
from kernel.sdk import SDKStore, Entity, Field, Identity, Relationship
from kernel.sdk.compile import compile_schema_from_classes
from kernel.sdk.dsl.expr import LogicVar, Pred
from kernel.sdk.dsl.rule import Rule

start = time.time()

## 1. Schema: Vendor Dependency Graph

In [8]:
class Vendor(Entity):
    vendor_id: str = Identity(primary_key=True)
    at_risk_signal: str = Field(cardinality="single")
    contingency_gap_signal: str = Field(cardinality="single")

class VendorDependency(Relationship):
    from_entity = Vendor
    to_entity = Vendor
    critical_path: str = Field(cardinality="single")

schema_ir = compile_schema_from_classes([Vendor, VendorDependency])
print(f"[{time.time()-start:.1f}s] Schema: {len(schema_ir['predicates'])} predicates")

[0.0s] Schema: 5 predicates


## 2. Session: Boolean Seeds + Side-Channel Assessments

Severity and exit-readiness bands are kept as side-channel context,
**not** fed into PyReason as bounded seeds.

Scenario seeded in the next cell:
- `ACME_CLOUD` starts with `at_risk_signal=true`
- `PAYMENTS_GATEWAY` starts with `contingency_gap_signal=true`
- critical-path edges form `ACME_CLOUD → PAYMENTS_GATEWAY → MOBILE_BANKING_APP`


In [9]:
session = PyReasonSession(schema_ir)

dependency_edges = [
    ("ACME_CLOUD", "PAYMENTS_GATEWAY"),
    ("PAYMENTS_GATEWAY", "MOBILE_BANKING_APP"),
]

side_channel_assessments = {
    "ACME_CLOUD": {"incident_severity_band": [0.80, 0.95], "exit_readiness_band": [0.45, 0.60]},
    "PAYMENTS_GATEWAY": {"incident_severity_band": [0.55, 0.70], "exit_readiness_band": [0.30, 0.50]},
    "MOBILE_BANKING_APP": {"incident_severity_band": [0.65, 0.85], "exit_readiness_band": [0.75, 0.90]},
}

with session.batch() as tx:
    acme = tx.entity(Vendor, vendor_id="ACME_CLOUD")
    payments = tx.entity(Vendor, vendor_id="PAYMENTS_GATEWAY")
    mobile = tx.entity(Vendor, vendor_id="MOBILE_BANKING_APP")

    acme.at_risk_signal.set("true", bound=[1.0, 1.0],
        meta={"source": "major_incident_triage", "mode": "boolean_seed"})
    payments.contingency_gap_signal.set("true", bound=[1.0, 1.0],
        meta={"source": "contract_review", "mode": "boolean_seed"})

    tx.relationship(VendorDependency, from_entity=acme, to_entity=payments,
        critical_path="true", bound=[1.0, 1.0], meta={"source": "dependency_mapping"})
    tx.relationship(VendorDependency, from_entity=payments, to_entity=mobile,
        critical_path="true", bound=[1.0, 1.0], meta={"source": "dependency_mapping"})
    tx.commit()

print(f"[{time.time()-start:.1f}s] Seeded vendors: {', '.join(sorted(side_channel_assessments))}")
print(f"  critical_path edges: {len(dependency_edges)}")
print(f"  annotation templates: {len(session.annotation_templates)}")


[0.0s] Seeded vendors: ACME_CLOUD, MOBILE_BANKING_APP, PAYMENTS_GATEWAY
  critical_path edges: 2
  annotation templates: 20


## 3. Rules: Boolean Risk Propagation

Two one-hop propagation rules are declared in the next cell:
1. `at_risk_signal(Y) + critical_path(Y,X) -> at_risk_signal(X)`
2. `contingency_gap_signal(Y) + critical_path(Y,X) -> contingency_gap_signal(X)`

Both use `PyReasonRuleExt(timestep_delay=1)` so the graph evolves over timesteps.


In [10]:
x = LogicVar("x")
y = LogicVar("y")

rules = [
    Rule(id="vendor_risk_propagation", version="1.0",
        select=[Pred("vendor:at_risk_signal", x)],
        where=[Pred("vendor:at_risk_signal", y), Pred("vendor_dependency:critical_path", y, x)],
        engine_ext=PyReasonRuleExt(timestep_delay=1)),
    Rule(id="contingency_gap_propagation", version="1.0",
        select=[Pred("vendor:contingency_gap_signal", x)],
        where=[Pred("vendor:contingency_gap_signal", y), Pred("vendor_dependency:critical_path", y, x)],
        engine_ext=PyReasonRuleExt(timestep_delay=1)),
]

print("rule_ids:", [rule.id for rule in rules])


rule_ids: ['vendor_risk_propagation', 'contingency_gap_propagation']


## 4. Run PyReason

If PyReason is unavailable, the next cell prints the actual runtime exception.

Expected propagation shape for a successful run:
- `at_risk_signal`: `ACME_CLOUD -> PAYMENTS_GATEWAY -> MOBILE_BANKING_APP`
- `contingency_gap_signal`: `PAYMENTS_GATEWAY -> MOBILE_BANKING_APP`


In [11]:
try:
    result = run_pyreason(session, rule_defs=rules,
        config=PyReasonRunConfig(timesteps=4, atom_trace=True))
except Exception as exc:
    result = None
    print(f"PyReason unavailable: {type(exc).__name__}: {exc}")


Added  0 graph-attribute node facts and  2 graph_attribute edge facts.
Filtering rules based on queries
Timestep: 0
Timestep: 1
Timestep: 2
Timestep: 3
Timestep: 4

Converged at time: 4
Fixed Point iterations: 5


## 5. Results: Active Labels per Timestep

In [12]:
if result is not None:
    interp = result.interpretation.get_dict()
    print(f"[{time.time()-start:.1f}s] Complete ({result.elapsed_seconds:.1f}s)")
    print(f"  Derived: {len(result.derived_session.node_facts)} node, {len(result.derived_session.edge_facts)} edge")

    # Interpretation keys mix node strings and edge tuples; sort by string form so the
    # display order is stable without comparing heterogeneous types directly.
    for t in sorted(interp.keys()):
        active = []
        for comp, preds in sorted(interp[t].items(), key=lambda kv: str(kv[0])):
            labels = sorted(p for p, (lo, hi) in preds.items() if lo == 1.0 and hi == 1.0)
            if labels: active.append(f"{comp}: {', '.join(labels)}")
        if active:
            print(f"\n  Timestep {t}:")
            for line in active: print(f"    {line}")

[0.2s] Complete (0.1s)
  Derived: 3 node, 0 edge

  Timestep 0:
    ACME_CLOUD: at_risk_signal
    PAYMENTS_GATEWAY: contingency_gap_signal

  Timestep 1:
    ACME_CLOUD: at_risk_signal
    MOBILE_BANKING_APP: contingency_gap_signal
    PAYMENTS_GATEWAY: at_risk_signal, contingency_gap_signal

  Timestep 2:
    ACME_CLOUD: at_risk_signal
    MOBILE_BANKING_APP: at_risk_signal, contingency_gap_signal
    PAYMENTS_GATEWAY: at_risk_signal, contingency_gap_signal

  Timestep 3:
    ACME_CLOUD: at_risk_signal
    MOBILE_BANKING_APP: at_risk_signal, contingency_gap_signal
    PAYMENTS_GATEWAY: at_risk_signal, contingency_gap_signal

  Timestep 4:
    ACME_CLOUD: at_risk_signal
    MOBILE_BANKING_APP: at_risk_signal, contingency_gap_signal
    PAYMENTS_GATEWAY: at_risk_signal, contingency_gap_signal


## 6. Accept + Annotations

In [13]:
if result is not None:
    target_sdk = SDKStore([Vendor])
    accept_result = accept_pyreason_session(target_sdk.ledger, result.derived_session)
    print(f"[{time.time()-start:.1f}s] Accepted: {len(accept_result.node_asrt_ids)} assertions, "
          f"{accept_result.annotation_count} annotations")

[0.2s] Accepted: 3 assertions, 9 annotations


---

**Runtime explain surface:** PyReason candidates use `explain-timeline` / `explain-timeline-summary` / `explain-timeline-narrative`
and the polymorphic `explain-summary` / `explain-narrative` / `explain-nl` candidate routes.
They do **not** use `explain-tree`, which remains the CandidateEvidenceTree surface for native / Souffle / ProbLog.

**Next:** [06_problog_probabilistic.ipynb](06_problog_probabilistic.ipynb) — ProbLog probabilistic reasoning
